In [1]:
import numpy as np
from itertools import product, combinations

In [2]:
# inputs
rng = np.random.default_rng(2026)
n, K, q = 6, 3, 0.5               # stocks, pick exactly K, risk aversion
mu = rng.uniform(0.05, 0.25, n)   # expected returns
A = rng.normal(0, 0.1, (n, n))
sigma = A @ A.T                   # covariance (risk)

In [3]:
A

array([[-0.03119493,  0.03038354, -0.02676603, -0.02259089,  0.07200678,
         0.05147052],
       [-0.00641279, -0.00854766,  0.01609163, -0.06140184, -0.04037503,
         0.05482602],
       [-0.01304828, -0.13744262, -0.04772787,  0.06566216, -0.02322828,
        -0.01487328],
       [ 0.06418366,  0.18246103, -0.07131887,  0.13482068, -0.12300128,
         0.01749776],
       [-0.11695295,  0.13514582,  0.0833923 ,  0.11377168, -0.08855332,
         0.0684555 ],
       [-0.05190133, -0.04573853,  0.05065371,  0.08767182,  0.02044202,
        -0.06279871]])

In [4]:
mu

array([0.08578696, 0.17798263, 0.14345368, 0.12410011, 0.12098347,
       0.20810365])

In [5]:
# create 20 states
stocks_lists = []
for stocks in combinations(range(n), K):
    x = np.zeros(n, dtype=int)
    x[list(stocks)] = 1
    stocks_lists.append(x)
    
stocks_lists[:3]

[array([1, 1, 1, 0, 0, 0]),
 array([1, 1, 0, 1, 0, 0]),
 array([1, 1, 0, 0, 1, 0])]

In [6]:
len(stocks_lists)

20

In [7]:
np.array(stocks_lists).sum(axis = 1)
# exactly three stocks are selected

array([3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3])

In [8]:
lam = 0.5 # initialised randomly

In [9]:
# create the cost function per state
def cost(x):
    return q * x @ sigma @ x - mu @ x + lam * (x.sum() - K) ** 2
  
cost(stocks_lists[0])

np.float64(-0.3931699692080152)

In [10]:
costs = []
for stock in stocks_lists:
  costs.append(cost(stock))
  
np.argmin(np.array(costs))

np.int64(12)

In [11]:
costs[12] # returns

np.float64(-0.5030146902647753)

In [12]:
stocks_lists[12]

array([0, 1, 1, 0, 0, 1])

In [13]:
# get all indices with 1
best_folio = stocks_lists[12]
best_stocks = []
for i, buy in enumerate(best_folio):
  if buy == 1:
    best_stocks.append(i)
    
best_stocks

[1, 2, 5]

In [14]:
var_best_within = np.array([sigma[i,i] for i in best_stocks]).sum()
var_best_within

np.float64(0.054589776532352155)

In [15]:
returns = np.array([mu[i] for i in best_stocks]).sum()
returns

np.float64(0.5295399625423809)

In [16]:
paired_combinations = list(combinations(best_stocks,2))

In [17]:
paired_combinations

[(1, 2), (1, 5), (2, 5)]

In [18]:
paired_var = np.array([sigma[i, j] for i, j in paired_combinations]).sum() * 2
paired_var

np.float64(-0.0015392319771409015)

In [19]:
total_var = var_best_within - paired_var

In [20]:
total_var

np.float64(0.05612900850949305)

In [21]:
risk = q * total_var
risk

np.float64(0.028064504254746526)

In [22]:
final_cost = risk - returns
final_cost

np.float64(-0.5014754582876344)